In [ ]:
# NOTEBOOK NAME
# KmeansSandbox.ipynb (SINGLE NOTEBOOK)
# NOTEBOOK NAME

# OPENING IMPORTS

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *

import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# # mapping things
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# import matplotlib.colors as mcolors
# import matplotlib.cm as cm
import matplotlib.patches as mpatches

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # for adding a colourful topo base map to the CAPI plots
# from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

# from matplotlib.patches import Circle # for radar range ring circles on the map
# from matplotlib.lines import Line2D # for plotting stars on the map legend


# # # for projecting radar coordinates to lat and lon
# # from pyproj import Geod

# # import matplotlib.dates as mdates # for putting dates on the time axis of plots

# from matplotlib.collections import LineCollection # plotting many lines at once

# sys.path.append('/home/563/sg3241/Notebooks/CustomFeatureTracking')
# from CustomTracking import *

# from scipy.stats import gaussian_kde # for plotting kernel density plots

# KMeans clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler # normalising data for distance calculations

In [ ]:
# FeatureStatsPath = '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/22/V5/20240309/22_20240309FeatureStats.nc'

FeatureStatsPath = '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/22/V5/QC2/20240214/trackstats_20240214.000000_20240214.235500.nc'
FeatureStatsXR = xr.open_dataset(FeatureStatsPath)



In [ ]:
FeatureStatsXR

In [ ]:
FeatureLifetimeMins = np.array(FeatureStatsXR['track_duration'].values) * 5         # numpy array for feature lifetimes in minutes
MeanETH_10DbZ = np.nanmean(np.array(FeatureStatsXR['maxETH_10dbz'].values),1)       # numpy array for mean echo top height at 20 dbz over feature lifetime
MeanMaxDbZ = np.nanmean(np.array(FeatureStatsXR['max_dbz'].values),1)               # numpy array for mean maximum reflectivity over feature lifetime
MeanArea = np.nanmean(np.array(FeatureStatsXR['core_area'].values),1)               # numpy array for mean core (convective) area over feature lifetime

# Stack as columns — each array becomes a feature column
data = np.column_stack([MeanETH_10DbZ, MeanMaxDbZ, MeanArea])

In [ ]:
# make each feature have mean=0 and std=1
Scaler = StandardScaler()
DataScaled = Scaler.fit_transform(data)


In [ ]:
KM = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
KM.fit(DataScaled)

# get the cluster label for each of your 5570 features
ClusterLabels = KM.labels_
print(ClusterLabels)        # e.g. [0, 2, 1, 4, 0, 3, ...]
print(ClusterLabels.shape)  # (5570,)

In [ ]:
# CHAD PLOT
# ALL 3 VARIABLES
# ECHO TOP HEIGHT, DBZ, AREA

Fig = plt.figure(figsize=(10, 8))
Ax  = Fig.add_subplot(111, projection='3d')

# get unique clusters and assign a colour to each
UniqueClusters = np.unique(ClusterLabels)
Cmap           = plt.cm.get_cmap('tab10', len(UniqueClusters))

for i, Cluster in enumerate(UniqueClusters):
    Mask = (ClusterLabels == Cluster)
    Ax.scatter(
        MeanArea[Mask],
        MeanMaxDbZ[Mask],
        MeanETH_10DbZ[Mask],
        color=Cmap(i),
        s=10,
        alpha=0.6,
        label=f'Cluster {Cluster}'
    )

Ax.set_xlim(0,  140)  # area in km^2
Ax.set_ylim(15, 55)   # reflectivity in dBZ
Ax.set_zlim(0,  12)   # ETH in kilometres

Ax.set_xlabel('Mean Area (km^2)')
Ax.set_ylabel('Mean Max dBZ')
Ax.set_zlabel('Mean ETH 10dBZ (km)')
Ax.set_title('K-Means Clustering of Mackay Feature Tracks 2024-03-09')

Ax.legend(title='Cluster', bbox_to_anchor=(1.15, 1), loc='upper left')
plt.tight_layout()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
SaveFile   = 'KmeansClusteringFeatures_22_20240214_3D.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)


In [ ]:
# CHAD PLOT
# 2 VARIABLES
# ECHO TOP HEIGHT, AREA

Fig, Ax = plt.subplots(figsize=(10, 8))

# get unique clusters and assign a colour to each
UniqueClusters = np.unique(ClusterLabels)
Cmap           = plt.cm.get_cmap('tab10', len(UniqueClusters))

for i, Cluster in enumerate(UniqueClusters):
    Mask = ClusterLabels == Cluster
    Ax.scatter(
        MeanArea[Mask],
        MeanETH_10DbZ[Mask],
        color=Cmap(i),
        s=10,
        alpha=0.6,
        label=f'Cluster {Cluster}'
    )

Ax.set_xlabel('Mean Area (km^2)')
Ax.set_ylabel('Mean ETH 10dBZ (km)')
Ax.set_title('K-Means Clustering of Mackay Feature Tracks 2024-03-09')

Ax.set_xlim(0,  140)  # area in km^2
Ax.set_ylim(0,  12)   # ETH in kilometres

plt.grid()

Ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
SaveFile   = 'KmeansClusteringFeatures_22_20240214_2DA.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
# CHAD PLOT
# 2 VARIABLES
# DBZ, AREA

Fig, Ax = plt.subplots(figsize=(10, 8))

# get unique clusters and assign a colour to each
UniqueClusters = np.unique(ClusterLabels)
Cmap           = plt.cm.get_cmap('tab10', len(UniqueClusters))

for i, Cluster in enumerate(UniqueClusters):
    Mask = ClusterLabels == Cluster
    Ax.scatter(
        MeanArea[Mask],
        MeanMaxDbZ[Mask],
        color=Cmap(i),
        s=10,
        alpha=0.6,
        label=f'Cluster {Cluster}'
    )

Ax.set_xlabel('Mean Area (km^2)')
Ax.set_ylabel('Mean Max dBZ')
Ax.set_title('K-Means Clustering of Mackay Feature Tracks 2024-03-09')

Ax.set_xlim(0,  140)  # area in km^2
Ax.set_ylim(15, 55)   # reflectivity in dBZ

plt.grid()

Ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()


SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
SaveFile   = 'KmeansClusteringFeatures_22_20240214_2DB.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)


In [ ]:
# CHAD PLOT
# 2 VARIABLES
# ECHO TOP HEIGHT, DBZ

Fig, Ax = plt.subplots(figsize=(10, 8))

# get unique clusters and assign a colour to each
UniqueClusters = np.unique(ClusterLabels)
Cmap           = plt.cm.get_cmap('tab10', len(UniqueClusters))

for i, Cluster in enumerate(UniqueClusters):
    Mask = ClusterLabels == Cluster
    Ax.scatter(
        MeanMaxDbZ[Mask],
        MeanETH_10DbZ[Mask],
        color=Cmap(i),
        s=10,
        alpha=0.6,
        label=f'Cluster {Cluster}'
    )

Ax.set_xlabel('Mean Max dBZ')
Ax.set_ylabel('Mean ETH 10dBZ (km)')
Ax.set_title('K-Means Clustering of Mackay Feature Tracks 2024-03-09')

Ax.set_xlim(15, 55)   # reflectivity in dBZ
Ax.set_ylim(0,  12)   # ETH in kilometres

plt.grid()

Ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
SaveFile   = 'KmeansClusteringFeatures_22_20240214_2DC.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
# CHAD
# GENERATE FAKE DATA TO TEST THE METHOD

np.random.seed(49)

# Cluster 1 — tight, small cluster (bottom left)
N1    = 200
C1_X  = np.random.normal(loc=10,  scale=6,  size=N1)
C1_Y  = np.random.normal(loc=10,  scale=4,  size=N1)

# Cluster 2 — large, spread out cluster (top left)
N2    = 2500
C2_X  = np.random.normal(loc=15,  scale=16,  size=N2)
C2_Y  = np.random.normal(loc=60,  scale=12, size=N2)

# Cluster 3 — medium cluster (bottom right)
N3    = 700
C3_X  = np.random.normal(loc=70,  scale=10,  size=N3)
C3_Y  = np.random.normal(loc=15,  scale=10,  size=N3)

# Cluster 4 — tight, small cluster (top right)
N4    = 1000
C4_X  = np.random.normal(loc=75,  scale=9,  size=N4)
C4_Y  = np.random.normal(loc=75,  scale=7,  size=N4)

# stitch together into one data matrix (5000, 2)
FakeX  = np.concatenate([C1_X, C2_X, C3_X, C4_X])
FakeY  = np.concatenate([C1_Y, C2_Y, C3_Y, C4_Y])
FakeData = np.column_stack([FakeX, FakeY])

print(FakeData.shape)  # should print (5000, 2)


In [ ]:
TestKM = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=7)
TestKM.fit(FakeData)

# get the cluster label for each of your 5570 features
TestClusterLabels = TestKM.labels_
print(TestClusterLabels)        # e.g. [0, 2, 1, 4, 0, 3, ...]
print(TestClusterLabels.shape)  # (5570,)

In [ ]:
# CHAD
# Clustering Run on 2D scatter plot of fake data

Fig, Ax = plt.subplots(figsize=(10, 8))

# get unique clusters and assign a colour to each
UniqueTestClusters = np.unique(TestClusterLabels)
TestCmap           = plt.cm.get_cmap('tab10', len(UniqueTestClusters))

for i, TestCluster in enumerate(UniqueTestClusters):
    Mask = TestClusterLabels == TestCluster
    Ax.scatter(
        FakeX[Mask],
        FakeY[Mask],
        color=TestCmap(i),
        s=10,
        alpha=0.6,
        label=f'Cluster {TestCluster}'
    )

Ax.set_xlabel('FakeX')
Ax.set_ylabel('FakeY')
Ax.set_title('K-Means Clustering of Fake Data')

Ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

plt.grid()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
SaveFile   = 'KmeansClusteringExample.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)